# Project 4: Survey Response Analyser

**The scenario:** Your company ran an internal satisfaction survey. 40 employees submitted a rating (1–5) and a free-text comment. HR needs a report: which departments are struggling, what topics come up most, and which responses need urgent attention.

Right now someone reads 40 comments by hand and writes a summary. You are going to automate it.

**What this connects to:** In future modules on machine learning and NLP, you will process text at scale — thousands or millions of documents. The operations here — splitting text into words, counting frequencies, filtering by value — are the exact same operations, just applied to 40 rows instead of 40 million.

**What you will build:**
1. Load and explore the survey data with pandas
2. Calculate average scores per department and flag struggling ones
3. Count the most frequent words across all comments
4. Compare word frequencies in low vs high-scoring responses
5. Export a summary report

**Concepts practised:** pandas, string methods, dictionaries as counters, functions, sorting, boolean filtering

---

## Step 1 — Load and explore the data

Load with pandas, then run the three standard inspection calls.
You should do this every time you open a new dataset — before writing a single line of analysis.

In [1]:
import pandas as pd

df = pd.read_csv('survey_responses.csv')

print(f'Shape: {df.shape}')
print()
df.head()

Shape: (40, 4)



,response_id,department,rating,comment
0,R001,Sales,5,The new CRM has completely changed how we work...
1,R002,Marketing,4,Good tools overall. Would love better integrat...
2,R003,HR,2,Onboarding process is still very manual. We ke...
3,R004,IT,5,Infrastructure upgrades this quarter were smoo...
4,R005,Sales,3,CRM is fine but training was too short. Half t...


In [2]:
print('Column types:')
print(df.dtypes)
print()
print('Missing values:')
print(df.isnull().sum())
print()
print('Responses per department:')
print(df['department'].value_counts())

Column types:
response_id      str
department       str
rating         int64
comment          str
dtype: object

Missing values:
response_id    0
department     0
rating         0
comment        0
dtype: int64

Responses per department:
department
Sales        9
Marketing    8
HR           8
IT           8
Finance      7
Name: count, dtype: int64


In [3]:
# Rating distribution — how many 1s, 2s, 3s, 4s, 5s?
print('Rating distribution:')
print(df['rating'].value_counts().sort_index())

Rating distribution:
rating
1     5
2     7
3     9
4    10
5     9
Name: count, dtype: int64


---
## Step 2 — Clean the ratings

Ratings should be integers between 1 and 5. Surveys sometimes contain blanks,
text instead of numbers, or out-of-range values like 0 or 6.

We write a function to handle all of it, then apply it to the whole column at once.

In [9]:
def parse_rating(value):
    """
    Converts a raw rating value to an integer.
    Returns None if missing, not a number, or outside 1-5.
    """
    if pd.isnull(value):
        return None
    try:
        rating = int(value)
    except (ValueError, TypeError):
        return None
    if rating < 1 or rating > 5:
        return None
    return rating


# Apply the function to every value in the 'rating' column
# This replaces the column with cleaned values
df['rating'] = df['rating'].apply(parse_rating)

skipped = df['rating'].isnull().sum()
print(f'Ratings cleaned. Invalid / missing: {skipped}')
print()

# Visual distribution with a simple bar (this is just text-based, not a real chart...we'll do real charts later!)
print('Rating distribution:')
for rating, count in df['rating'].value_counts().sort_index().items():
    bar = '=' * count
    print(f'  {rating} ★  {bar} ({count})')

Ratings cleaned. Invalid / missing: 0

Rating distribution:
  1 ★  ===== (5)
  2 ★  ======= (7)
  3 ★  ========= (9)
  4 ★  ========== (10)
  5 ★  ========= (9)


---
## Step 3 — Average score per department

You have used `groupby` before. This time you write it yourself.

Then flag departments whose average falls below a threshold —
exactly the same boolean filtering pattern from Projects 1 and 2.

In [ ]:
# Work only with rows that have a valid rating
df_valid = df[df['rating'].notna()].copy()

# YOUR CODE HERE
# Calculate average rating and response count per department using groupby + agg
# Sort from lowest average to highest (problems appear at the top)
# Round to 2 decimal places
# Store the result in dept_summary


In [ ]:
ALERT_THRESHOLD = 3.0

# YOUR CODE HERE
# Use boolean filtering on dept_summary to find departments below the threshold
# Print a clear alert for each one — e.g. "ALERT: Finance — avg rating 2.17"


---
## Step 4 — Word frequency analysis

This is the new piece — and the most important for your future modules.

**The idea:** take every comment, split it into individual words, count how many times
each word appears. The result is a **word frequency dictionary** — one of the most
fundamental data structures in text analysis and machine learning.

Before any ML model reads text, it almost always starts here: convert text into counts.  
What you build now is called a **bag of words** — the foundation of text classification.

**Steps:**
1. Lowercase everything
2. Split into individual words
3. Remove common filler words ('the', 'a', 'is'...) — called **stopwords**
4. Count how many times each meaningful word appears across all comments

In [6]:
# Common English words that carry no useful meaning — we ignore these
# In professional NLP libraries this list has thousands of entries
STOPWORDS = {
    'the', 'a', 'an', 'and', 'or', 'but', 'in', 'on', 'at', 'to',
    'for', 'of', 'with', 'is', 'are', 'was', 'were', 'has', 'have',
    'had', 'be', 'been', 'this', 'that', 'it', 'its', 'we', 'our',
    'us', 'by', 'from', 'as', 'not', 'no', 'so', 'if', 'do', 'did',
    'i', 'my', 'me', 'he', 'she', 'they', 'their', 'there', 'about',
    'more', 'very', 'too', 'also', 'still', 'much', 'some', 'all',
    'can', 'will', 'would', 'could', 'should', 'been', 'into', 'up',
    'how', 'what', 'which', 'who', 'when', 'than', 'just', 'now'
}


def tokenise(text):
    """
    Converts a comment string into a list of meaningful words.
    Lowercases, splits on whitespace, strips punctuation, removes stopwords.
    """
    words = str(text).lower().split()
    clean = []
    for word in words:
        word = word.strip('.,!?;:\'"()')
        if word and word not in STOPWORDS and len(word) >= 3:
            clean.append(word)
    return clean


# Test on a single comment
sample = df['comment'].iloc[0]
print('Original:')
print(' ', sample)
print()
print('After tokenising:')
print(' ', tokenise(sample))

Original:
  The new CRM has completely changed how we work. Huge time saver and the reporting is excellent.

After tokenising:
  ['new', 'crm', 'completely', 'changed', 'work', 'huge', 'time', 'saver', 'reporting', 'excellent']


In [8]:
def count_words(series):
    """
    Takes a pandas Series of comment strings.
    Returns a word frequency dictionary: {word: count}.
    """
    word_counts = {}
    for comment in series:
        for word in tokenise(comment):
            word_counts[word] = word_counts.get(word, 0) + 1
    return word_counts


def top_words(word_counts, n=15):
    """
    Returns the n most frequent (word, count) pairs, sorted descending.
    """
    return sorted(word_counts.items(), key=lambda item: item[1], reverse=True)[:n]


# Count words across ALL comments
all_word_counts = count_words(df['comment'])

print(f'Unique words found: {len(all_word_counts)}')
print()
print('Top 15 most frequent words:')
for word, count in top_words(all_word_counts):
    bar = '=' * count
    print(f'  {word:<22} {bar} ({count})')

Unique words found: 264

Top 15 most frequent words:
  new                    ========= (9)
  quarter                ======= (7)
  team                   ====== (6)
  month                  ====== (6)
  good                   ===== (5)
  process                ==== (4)
  communication          ==== (4)
  improved               ==== (4)
  well                   ==== (4)
  work                   === (3)
  excellent              === (3)
  tools                  === (3)
  better                 === (3)
  training               === (3)
  system                 === (3)


---
## Step 5 — Low-rated vs high-rated: what are people saying?

Instead of counting words across all comments, we count separately for responses
rated 1–2 versus 4–5. Words that appear in unhappy comments but not in happy ones
are signals — they tell you what is going wrong.

This is the exact same logic a sentiment analysis model uses before training.

In [ ]:
# Split into two groups using boolean filtering — a pattern you know well
low_comments  = df_valid[df_valid['rating'] <= 2]['comment']
high_comments = df_valid[df_valid['rating'] >= 4]['comment']

print(f'Low-rated responses (1-2):  {len(low_comments)}')
print(f'High-rated responses (4-5): {len(high_comments)}')
print()

# YOUR CODE HERE
# Use count_words() on each group, then top_words() to print the top 10 for each
# Print them one after the other with a clear heading
#
# Expected output format:
#
# Top words in LOW-rated comments:
#   process                ████ (4)
#   ...
#
# Top words in HIGH-rated comments:
#   team                   ████████ (8)
#   ...

**What do you notice?**

Compare the two word lists. Do the low-rated comments cluster around specific topics?
Do the high-rated ones use different vocabulary?

This manual comparison is what a **text classifier** learns to do automatically
from thousands of labelled examples. You are doing it by hand first to build intuition.

---

## Step 6 — Export the report

Two output files:
1. `survey_dept_summary.csv` — average rating and response count per department
2. `survey_flagged_responses.csv` — full text of responses rated 1 or 2, for HR to review

In [ ]:
# Export 1: department summary — you write this

# YOUR CODE HERE
# Save dept_summary to 'survey_dept_summary.csv'
# Use .to_csv() — one line


# Export 2: flagged low-rated responses
flagged = df_valid[df_valid['rating'] <= 2][['response_id', 'department', 'rating', 'comment']]
flagged.to_csv('survey_flagged_responses.csv', index=False)
print(f'Saved: survey_flagged_responses.csv  ({len(flagged)} rows)')
print()
print(flagged.to_string(index=False))

---
## Step 7 — Reflect

**Discussion questions:**
1. The word frequency analysis finds which words appear most — but not *why*. What is missing compared to a real sentiment model?
2. Why did we lowercase all words before counting? What would happen if we didn't?
3. How would you improve the stopword list? What other words could be removed?
4. If you had 10,000 comments instead of 40, would this script still work?

**Extension A — per-department word frequency:**  
For each department, find the 5 most common words in their comments.  
Do different departments talk about different topics?

In [ ]:
# Extension A: top 5 words per department
# YOUR CODE HERE
# Hint: loop over df['department'].unique()
# For each department, filter df, pass the 'comment' column to count_words(), then top_words(5)


**Extension B — rating trend:**  
Response IDs go R001 → R040, roughly reflecting submission order.  
Did sentiment improve or decline as the survey progressed?

In [ ]:
# Extension B: rating trend
# YOUR CODE HERE
# Hint: df_valid['response_num'] = df_valid['response_id'].str[1:].astype(int)
# Then groupby('response_num') or just sort and print the rolling average


---
## Concepts reinforced

| Concept | Where you used it |
|---|---|
| `pd.read_csv()` | Loading the dataset in one line |
| `.apply()` | Applying `parse_rating()` to the whole column |
| `groupby` + `agg` | Department averages and counts |
| Boolean filtering | Separating low / high responses, flagging departments |
| `.to_csv()` | Exporting results |
| Functions + try/except | `parse_rating()` — safe conversion |
| Dictionaries as counters | `count_words()` — word frequency table |
| String methods | `.lower()`, `.split()`, `.strip()` — text preprocessing |
| Sorting with a lambda | `sorted(..., key=lambda item: item[1])` |

**Connection to future modules:**

| What you did here | What it becomes later |
|---|---|
| Word frequency dictionary | Bag-of-words feature representation |
| Stopword removal | Text preprocessing pipeline |
| Low vs high word comparison | Feature selection for classification |
| `tokenise()` | Tokenisation step before NLP models |
| Rating as a label | Target variable in supervised learning |